# Data Preprocessing


## Data loading

#### Hold-on split: (1) 2024-06-01 (80% + 20%); (2) 2024-10-01; (3) 2025-01-01

In [1]:
import os
import pandas as pd

class CFG:
    # --- Data & Feature Parameters ---
    COIN_ID_COLUMN        = 'coin_id'
    TIMESTAMP_COLUMN      = 'timestamp'
    TARGET_COLUMN         = 'target_direction'

    # --- Split & CV Parameters for ratio-based hold-out ---
    TRAIN_RATIO           = 0.8     # proportion of data for training
    SPLIT_ROUND_FREQUENCY = 'month' # how to round the cutoff: 'month', 'day', or '' for no rounding

def load_data(path: str) -> pd.DataFrame:
    """
    Load all coins from a partitioned Parquet dataset and
    ensure proper datetime conversion and sorting.
    - path: root folder of the parquet dataset (with coin_id=... subfolders)
    Returns a DataFrame with 'coin_id' and 'timestamp'.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Data file not found: {path}")

    df = pd.read_parquet(path, engine='pyarrow')

    # ensure timestamp column is datetime type
    if not pd.api.types.is_datetime64_any_dtype(df[CFG.TIMESTAMP_COLUMN]):
        df[CFG.TIMESTAMP_COLUMN] = pd.to_datetime(df[CFG.TIMESTAMP_COLUMN])

    # sort by coin_id then timestamp
    df.sort_values([CFG.COIN_ID_COLUMN, CFG.TIMESTAMP_COLUMN],
                   inplace=True, ignore_index=True)
    return df

def split_by_ratio(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Hold-out split based on TRAIN_RATIO.
    - calculates exact cutoff between min and max timestamp
    - optionally rounds cutoff down to month/day boundary
    Returns (train_df, test_df).
    """
    min_ts = df[CFG.TIMESTAMP_COLUMN].min()
    max_ts = df[CFG.TIMESTAMP_COLUMN].max()
    exact_cutoff = min_ts + (max_ts - min_ts) * CFG.TRAIN_RATIO

    # round cutoff if requested
    freq = CFG.SPLIT_ROUND_FREQUENCY.lower()
    if freq == 'month':
        cutoff = exact_cutoff.to_period('M').to_timestamp()
    elif freq == 'day':
        cutoff = exact_cutoff.normalize()
    else:
        cutoff = exact_cutoff

    # split into train and test
    train = df[df[CFG.TIMESTAMP_COLUMN] < cutoff].copy()
    test  = df[df[CFG.TIMESTAMP_COLUMN] >= cutoff].copy()

    print(f"[Ratio Split] exact_cutoff={exact_cutoff}, rounded_cutoff={cutoff}")
    print(f"  train period: {train[CFG.TIMESTAMP_COLUMN].min()} → {train[CFG.TIMESTAMP_COLUMN].max()}")
    print(f"  test  period: {test[CFG.TIMESTAMP_COLUMN].min()} → {test[CFG.TIMESTAMP_COLUMN].max()}")
    return train, test

def split_by_date(df: pd.DataFrame, date_str: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Hold-out split at a fixed date.
    - date_str: ISO format date (e.g. '2024-10-01')
    Returns (train_df, test_df).
    """
    cutoff = pd.to_datetime(date_str)
    train = df[df[CFG.TIMESTAMP_COLUMN] < cutoff].copy()
    test  = df[df[CFG.TIMESTAMP_COLUMN] >= cutoff].copy()

    print(f"[Date Split @ {date_str}]")
    print(f"  train period: {train[CFG.TIMESTAMP_COLUMN].min()} → {train[CFG.TIMESTAMP_COLUMN].max()}")
    print(f"  test  period: {test[CFG.TIMESTAMP_COLUMN].min()} → {test[CFG.TIMESTAMP_COLUMN].max()}")
    return train, test

if __name__ == "__main__":
    # load the full dataset
    data_path = os.path.join(os.getcwd(), "data", "OHLCV_ffill.parquet")
    df = load_data(data_path)

    # 1) 80% / 20% ratio split
    train1, test1 = split_by_ratio(df)

    # 2) fixed-date split at 2024-10-01
    train2, test2 = split_by_date(df, "2024-10-01")

    # 3) fixed-date split at 2025-01-01
    train3, test3 = split_by_date(df, "2025-01-01")

    # display shapes for quick sanity check
    print("=== Dataset Shapes ===")
    print(f"Ratio split    : train={train1.shape}, test={test1.shape}")
    print(f"Split @2024-10-01: train={train2.shape}, test={test2.shape}")
    print(f"Split @2025-01-01: train={train3.shape}, test={test3.shape}")

[Ratio Split] exact_cutoff=2024-06-15 16:48:00, rounded_cutoff=2024-06-01 00:00:00
  train period: 2021-01-01 00:00:00 → 2024-05-31 23:59:00
  test  period: 2024-06-01 00:00:00 → 2025-04-27 03:00:00
[Date Split @ 2024-10-01]
  train period: 2021-01-01 00:00:00 → 2024-09-30 23:59:00
  test  period: 2024-10-01 00:00:00 → 2025-04-27 03:00:00
[Date Split @ 2025-01-01]
  train period: 2021-01-01 00:00:00 → 2024-12-31 23:59:00
  test  period: 2025-01-01 00:00:00 → 2025-04-27 03:00:00
=== Dataset Shapes ===
Ratio split    : train=(8978400, 14), test=(2376905, 14)
Split @2024-10-01: train=(9856800, 14), test=(1498505, 14)
Split @2025-01-01: train=(10519200, 14), test=(836105, 14)


## Resampling
#### Time intervals: 1-min, 10-min, 1-hour, 1-day 

In [6]:
import os
import pandas as pd

class CFG:
    # Data & Feature Parameters
    COIN_ID_COLUMN   = 'coin_id'
    TIMESTAMP_COLUMN = 'timestamp'

    # Ratio-split parameters
    TRAIN_RATIO           = 0.8     # 80% train, 20% test
    SPLIT_ROUND_FREQUENCY = 'month' # 'month', 'day', or '' for no rounding

def load_data(path: str) -> pd.DataFrame:
    """
    Load full Parquet dataset, convert timestamp, sort by coin_id+timestamp.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Data file not found: {path}")

    df = pd.read_parquet(path, engine='pyarrow')
    if not pd.api.types.is_datetime64_any_dtype(df[CFG.TIMESTAMP_COLUMN]):
        df[CFG.TIMESTAMP_COLUMN] = pd.to_datetime(df[CFG.TIMESTAMP_COLUMN])
    df.sort_values([CFG.COIN_ID_COLUMN, CFG.TIMESTAMP_COLUMN],
                   inplace=True, ignore_index=True)
    return df

def split_by_ratio(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    80/20 hold-out split, optional rounding to month/day boundary.
    """
    min_ts = df[CFG.TIMESTAMP_COLUMN].min()
    max_ts = df[CFG.TIMESTAMP_COLUMN].max()
    exact_cutoff = min_ts + (max_ts - min_ts) * CFG.TRAIN_RATIO

    freq = CFG.SPLIT_ROUND_FREQUENCY.lower()
    if freq == 'month':
        cutoff = exact_cutoff.to_period('M').to_timestamp()
    elif freq == 'day':
        cutoff = exact_cutoff.normalize()
    else:
        cutoff = exact_cutoff

    train = df[df[CFG.TIMESTAMP_COLUMN] < cutoff].copy()
    test  = df[df[CFG.TIMESTAMP_COLUMN] >= cutoff].copy()
    return train, test

def split_by_date(df: pd.DataFrame, date_str: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Fixed-date hold-out split at date_str (inclusive for test).
    """
    cutoff = pd.to_datetime(date_str)
    train = df[df[CFG.TIMESTAMP_COLUMN] < cutoff].copy()
    test  = df[df[CFG.TIMESTAMP_COLUMN] >= cutoff].copy()
    return train, test

def resample_ohlcv(df: pd.DataFrame, freq: str) -> pd.DataFrame:
    """
    Resample OHLCV for each coin_id at given freq.
    - df must have coin_id col and timestamp index
    """
    agg = {
        "open":                        "first",
        "high":                        "max",
        "low":                         "min",
        "close":                       "last",
        "volume":                      "sum",
        "quote_asset_volume":          "sum",
        "number_of_trades":            "sum",
        "taker_buy_base_asset_volume": "sum",
        "taker_buy_quote_asset_volume":"sum",
    }
    df = df.set_index("timestamp")
    out = (
        df
        .groupby("coin_id", observed=True)
        .resample(freq)
        .agg(agg)
        .dropna(subset=["open"])
        .reset_index()
    )
    return out

if __name__ == "__main__":
    # 1) Load data
    data_path = os.path.join(os.getcwd(), "data", "OHLCV_ffill.parquet")
    df = load_data(data_path)

    # 2) Generate splits
    train1, test1 = split_by_ratio(df)
    train2, test2 = split_by_date(df, "2024-10-01")
    train3, test3 = split_by_date(df, "2025-01-01")

    splits = [
        ("ratio",      train1, test1),
        ("2024-10-01", train2, test2),
        ("2025-01-01", train3, test3),
    ]

    # 3) Define frequencies
    freq_map = ["1min", "10min", "1h", "1d"]

    # 4) Loop over splits → train/test → freqs → resample & save
    base_out = os.path.join("data", "resampled-v2")
    for split_name, train_df, test_df in splits:
        split_dir = os.path.join(base_out, split_name)
        os.makedirs(split_dir, exist_ok=True)

        for label, subset in [("train", train_df), ("test", test_df)]:
            for freq in freq_map:
                print(f"Resampling split={split_name}, subset={label}, freq={freq} …")
                rs = resample_ohlcv(subset, freq)
                out_file = f"{label}_{freq}.parquet"
                out_path = os.path.join(split_dir, out_file)
                rs.to_parquet(out_path)
                print(f"  → saved {out_path} (rows={rs.shape[0]})")

Resampling split=ratio, subset=train, freq=1min …
  → saved data/resampled-v2/ratio/train_1min.parquet (rows=8978400)
Resampling split=ratio, subset=train, freq=10min …
  → saved data/resampled-v2/ratio/train_10min.parquet (rows=897840)
Resampling split=ratio, subset=train, freq=1h …
  → saved data/resampled-v2/ratio/train_1h.parquet (rows=149640)
Resampling split=ratio, subset=train, freq=1d …
  → saved data/resampled-v2/ratio/train_1d.parquet (rows=6235)
Resampling split=ratio, subset=test, freq=1min …
  → saved data/resampled-v2/ratio/test_1min.parquet (rows=2376905)
Resampling split=ratio, subset=test, freq=10min …
  → saved data/resampled-v2/ratio/test_10min.parquet (rows=237695)
Resampling split=ratio, subset=test, freq=1h …
  → saved data/resampled-v2/ratio/test_1h.parquet (rows=39620)
Resampling split=ratio, subset=test, freq=1d …
  → saved data/resampled-v2/ratio/test_1d.parquet (rows=1655)
Resampling split=2024-10-01, subset=train, freq=1min …
  → saved data/resampled-v2/2024